# RQ0 — Data Preparation & Shared Utilities (Mac M4 edition)

**Thesis:** *Causal Graph-Augmented Multimodal LLM Framework for Explainable Decision Intelligence (Retail Case Study)*
**Author:** Bhanu Teja Malineni
**Compute target:** Mac mini M4, 24 GB RAM (MPS backend)

This is the **data-prep pipeline** that every RQ notebook depends on. Run this first.

## What this notebook does
1. Loads the three raw datasets from your local filesystem:
   - **Amazon Reviews 2023** (Electronics subset: reviews + metadata) — streams the JSONL files to stay within RAM
   - **RetailRocket** (events, item properties, category tree)
   - **Instacart Market Basket** (orders, products, aisles, departments)
2. Cleans, standardizes, and integrates them (per §6.3 of the proposal).
3. Builds reusable feature tables:
   - user / product / session tables
   - Amazon text embeddings per product (sentence-transformers MiniLM, real text)
   - Instacart basket-level features
   - retail-variable daily panel (price, promo, visibility, demand, etc.) used by causal RQs
4. Creates four stress-test subsets for RQ6.
5. Writes everything to `./outputs/prepared/` so other notebooks can pick them up.

## Expected folder layout (before running)

```
<project-root>/
├── data/
│   └── raw/
│       ├── amazon/
│       │   ├── Electronics.jsonl
│       │   └── meta_Electronics.jsonl
│       ├── retailrocket/
│       │   ├── events.csv
│       │   ├── category_tree.csv
│       │   ├── item_properties_part1.csv
│       │   └── item_properties_part2.csv
│       └── instacart/
│           ├── orders.csv
│           ├── products.csv
│           ├── aisles.csv
│           ├── departments.csv
│           ├── order_products__prior.csv
│           └── order_products__train.csv
└── notebooks/
    └── RQ0_data_preparation.ipynb    <- this notebook
```

The notebook will create `outputs/prepared/`, `outputs/figures/`, `outputs/tables/` automatically.


In [1]:
# ============================================================
# 0.1  Preset + paths + reproducibility
# ============================================================
import os, sys, json, warnings, random, time, gc, logging
from pathlib import Path
import numpy as np
import pandas as pd

# ---- Suppress non-actionable library warnings for clean notebook output ----
# These are not errors. They appear because:
#   - sentence-transformers/MiniLM stores a position_ids tensor that newer
#     `transformers` versions don't use (forward-compat noise, not a bug).
#   - HuggingFace asks anonymous downloaders to set HF_TOKEN for higher rate
#     limits; we don't need higher limits for this one-time model download.
warnings.filterwarnings("ignore")
os.environ["TRANSFORMERS_VERBOSITY"]      = "error"   # hide the BertModel LOAD REPORT
os.environ["HF_HUB_DISABLE_TELEMETRY"]    = "1"
os.environ["TOKENIZERS_PARALLELISM"]      = "false"   # silence fork warning
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"]  = "1"  # silence symlink warning on macOS
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"]     = "0"  # keep progress bars visible
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"]    = "1"  # silence the HF_TOKEN warning
# Belt-and-suspenders: filter the specific UserWarning class HF Hub raises
warnings.filterwarnings("ignore", message=".*HF_TOKEN.*")
warnings.filterwarnings("ignore", message=".*unauthenticated requests.*")
for noisy in ["transformers", "sentence_transformers", "huggingface_hub",
              "huggingface_hub.utils", "urllib3", "filelock"]:
    logging.getLogger(noisy).setLevel(logging.ERROR)

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ---- Preset controls data-sample sizes. Change this one line to switch.
# "small"  = fast debug   (~3 min, ~2 GB RAM peak)
# "medium" = recommended  (~10 min, ~6-8 GB RAM peak)
# "large"  = thesis run   (~40 min, ~14-18 GB RAM peak)
PRESET = "medium"

PRESETS = {
    "small":  dict(n_amazon_reviews=50_000,   n_rr_events=500_000,  n_ic_orders=200_000,  n_amazon_products_embed=5_000),
    "medium": dict(n_amazon_reviews=300_000,  n_rr_events=3_000_000,n_ic_orders=1_000_000,n_amazon_products_embed=20_000),
    "large":  dict(n_amazon_reviews=1_500_000,n_rr_events=10_000_000,n_ic_orders=3_400_000,n_amazon_products_embed=80_000),
}
CFG = PRESETS[PRESET]
print(f"Preset = {PRESET}: {CFG}")

# ---- Paths (relative to where the notebook is launched)
PROJECT = Path.cwd()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent   # allow running from notebooks/ or project root
DATA_RAW = PROJECT / "data" / "raw"
OUT      = PROJECT / "outputs"
PREP     = OUT / "prepared"; PREP.mkdir(parents=True, exist_ok=True)
FIG      = OUT / "figures";  FIG.mkdir(parents=True, exist_ok=True)
TAB      = OUT / "tables";   TAB.mkdir(parents=True, exist_ok=True)

PATHS = {
    "amazon_reviews": DATA_RAW / "amazon" / "Electronics.jsonl",
    "amazon_meta":    DATA_RAW / "amazon" / "meta_Electronics.jsonl",
    "rr_events":      DATA_RAW / "retailrocket" / "events.csv",
    "rr_cat_tree":    DATA_RAW / "retailrocket" / "category_tree.csv",
    "rr_item_p1":     DATA_RAW / "retailrocket" / "item_properties_part1.csv",
    "rr_item_p2":     DATA_RAW / "retailrocket" / "item_properties_part2.csv",
    "ic_orders":      DATA_RAW / "instacart" / "orders.csv",
    "ic_products":    DATA_RAW / "instacart" / "products.csv",
    "ic_aisles":      DATA_RAW / "instacart" / "aisles.csv",
    "ic_departments": DATA_RAW / "instacart" / "departments.csv",
    "ic_op_prior":    DATA_RAW / "instacart" / "order_products__prior.csv",
    "ic_op_train":    DATA_RAW / "instacart" / "order_products__train.csv",
}

print("Project root:", PROJECT)
print("Prepared dir:", PREP)
for k, p in PATHS.items():
    print(f"  {k:20s} exists={p.exists()}  {p}")


Preset = medium: {'n_amazon_reviews': 300000, 'n_rr_events': 3000000, 'n_ic_orders': 1000000, 'n_amazon_products_embed': 20000}
Project root: /Users/bhanutejamalineni/Thesis
Prepared dir: /Users/bhanutejamalineni/Thesis/outputs/prepared
  amazon_reviews       exists=True  /Users/bhanutejamalineni/Thesis/data/raw/amazon/Electronics.jsonl
  amazon_meta          exists=True  /Users/bhanutejamalineni/Thesis/data/raw/amazon/meta_Electronics.jsonl
  rr_events            exists=True  /Users/bhanutejamalineni/Thesis/data/raw/retailrocket/events.csv
  rr_cat_tree          exists=True  /Users/bhanutejamalineni/Thesis/data/raw/retailrocket/category_tree.csv
  rr_item_p1           exists=True  /Users/bhanutejamalineni/Thesis/data/raw/retailrocket/item_properties_part1.csv
  rr_item_p2           exists=True  /Users/bhanutejamalineni/Thesis/data/raw/retailrocket/item_properties_part2.csv
  ic_orders            exists=True  /Users/bhanutejamalineni/Thesis/data/raw/instacart/orders.csv
  ic_products  

## 0.2 Device detection (MPS / CUDA / CPU)
On Mac M4 we default to MPS (Apple GPU). Each downstream RQ notebook redefines this inline to stay standalone.


In [2]:
try:
    import torch
    if torch.backends.mps.is_available():
        DEVICE = "mps"
    elif torch.cuda.is_available():
        DEVICE = "cuda"
    else:
        DEVICE = "cpu"
except Exception:
    DEVICE = "cpu"
print("Device:", DEVICE)


Device: mps


## 0.3 Shared plotting helpers


In [3]:
import matplotlib as mpl, matplotlib.pyplot as plt
mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.family": "DejaVu Sans", "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "pdf.fonttype": 42,
})
def save_fig(fig, name): p=FIG/f"{name}.pdf"; fig.savefig(p, format="pdf"); print(f"  saved -> {p}"); return p
def save_table(df, name): p=TAB/f"{name}.csv"; df.to_csv(p, index=False); print(f"  saved -> {p}"); return p


## 0.4 Load Amazon Reviews 2023 (streaming)

We stream the JSONL files line-by-line to keep RAM bounded.


In [4]:
def load_amazon_reviews(path, limit):
    if not path.exists():
        print(f"  [skip] Amazon reviews file not found: {path}")
        return None
    keep = ["user_id","parent_asin","rating","title","text","timestamp","verified_purchase","helpful_vote"]
    rows = []
    t0 = time.time()
    with open(path, "r") as f:
        for i, line in enumerate(f):
            if i >= limit: break
            r = json.loads(line)
            rows.append({k: r.get(k) for k in keep})
            if (i+1) % 100_000 == 0:
                print(f"  ... {i+1:,} reviews loaded ({time.time()-t0:.1f}s)")
    df = pd.DataFrame(rows)
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms", errors="coerce")
    df = df.dropna(subset=["user_id","parent_asin","rating"])
    return df

def load_amazon_meta(path):
    if not path.exists():
        print(f"  [skip] Amazon meta file not found: {path}")
        return None
    keep = ["parent_asin","title","main_category","categories","price",
            "average_rating","rating_number","features","description","store","images"]
    rows = []
    with open(path, "r") as f:
        for line in f:
            r = json.loads(line)
            rows.append({k: r.get(k) for k in keep})
    m = pd.DataFrame(rows)
    m["price"] = pd.to_numeric(m["price"], errors="coerce")
    return m

amz_rev  = load_amazon_reviews(PATHS["amazon_reviews"], CFG["n_amazon_reviews"])
amz_meta = load_amazon_meta(PATHS["amazon_meta"])
if amz_rev is not None and amz_meta is not None:
    print(f"Amazon reviews: {len(amz_rev):,}  |  meta: {len(amz_meta):,}")
    amz = amz_rev.merge(amz_meta, on="parent_asin", how="left", suffixes=("","_meta"))
    amz.to_parquet(PREP / "amazon_merged.parquet", index=False)
    print(f"  -> saved amazon_merged.parquet ({(PREP/'amazon_merged.parquet').stat().st_size/1e6:.1f} MB)")
else:
    amz = None


  ... 100,000 reviews loaded (0.3s)
  ... 200,000 reviews loaded (0.5s)
  ... 300,000 reviews loaded (0.7s)
Amazon reviews: 300,000  |  meta: 1,610,012
  -> saved amazon_merged.parquet (404.3 MB)


## 0.5 Amazon text embeddings (real, sentence-transformers MiniLM)

For each product, concatenate title + first few words of description and encode with MiniLM (≈80 MB download, first call). On M4/MPS, encoding ~20k products takes about 45–90 seconds.


In [5]:
def build_amazon_product_embeddings(amz_meta, n_products, out_path):
    if amz_meta is None: print("  skip: no Amazon meta"); return None
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError:
        print("  [install] pip install sentence-transformers")
        return None

    meta = amz_meta.dropna(subset=["parent_asin","title"]).copy()
    meta["rating_number"] = pd.to_numeric(meta["rating_number"], errors="coerce").fillna(0)
    meta = meta.sort_values("rating_number", ascending=False).head(n_products)

    def make_text(row):
        desc = row["description"]
        if isinstance(desc, list): desc = " ".join(str(x) for x in desc[:3])
        desc = str(desc)[:300] if desc else ""
        title = str(row.get("title") or "")
        return f"{title}. {desc}".strip(". ")
    texts = meta.apply(make_text, axis=1).tolist()

    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
    t0 = time.time()
    embs = model.encode(texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
    print(f"  encoded {len(texts):,} products in {time.time()-t0:.1f}s")

    out = pd.DataFrame(embs, columns=[f"e{i:03d}" for i in range(embs.shape[1])])
    out.insert(0, "parent_asin", meta["parent_asin"].values)
    out.to_parquet(out_path, index=False)
    print(f"  -> saved {out_path.name} ({out_path.stat().st_size/1e6:.1f} MB)")
    return out

amz_embs = build_amazon_product_embeddings(amz_meta, CFG["n_amazon_products_embed"], PREP / "amazon_product_embeddings.parquet")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

  encoded 20,000 products in 32.7s
  -> saved amazon_product_embeddings.parquet (43.4 MB)


## 0.6 Load RetailRocket (events + item properties)


In [6]:
def load_rr(n_events):
    if not PATHS["rr_events"].exists():
        print(f"  [skip] RetailRocket events not found: {PATHS['rr_events']}")
        return None, None, None
    ev = pd.read_csv(PATHS["rr_events"], nrows=n_events)
    ev["timestamp"] = pd.to_datetime(ev["timestamp"], unit="ms")
    cat = pd.read_csv(PATHS["rr_cat_tree"])
    ip = pd.concat([pd.read_csv(PATHS["rr_item_p1"]),
                    pd.read_csv(PATHS["rr_item_p2"])], ignore_index=True)
    ip["timestamp"] = pd.to_datetime(ip["timestamp"], unit="ms")
    return ev, cat, ip

rr_events, rr_cat, rr_item_props = load_rr(CFG["n_rr_events"])

if rr_events is not None:
    print(f"RR events: {len(rr_events):,} | item-props rows: {len(rr_item_props):,}")

    # Session inference: new session on >30-min gap
    rr_events = rr_events.sort_values(["visitorid","timestamp"])
    gap = rr_events.groupby("visitorid")["timestamp"].diff().dt.total_seconds()
    rr_events["new_session"] = ((gap.isna()) | (gap > 1800)).astype(int)
    rr_events["session_id"] = rr_events.groupby("visitorid")["new_session"].cumsum()
    rr_events["session_uid"] = rr_events["visitorid"].astype(str) + "_" + rr_events["session_id"].astype(str)

    # price-like property (790) and categoryid
    price_props = rr_item_props[rr_item_props["property"]=="790"].copy()
    price_props["price"] = pd.to_numeric(price_props["value"].str.extract(r"([0-9.]+)")[0], errors="coerce")
    cat_props = rr_item_props[rr_item_props["property"]=="categoryid"].copy()
    cat_props["categoryid"] = pd.to_numeric(cat_props["value"], errors="coerce")

    rr_events.to_parquet(PREP / "rr_events.parquet", index=False)
    rr_cat.to_parquet(PREP / "rr_cat.parquet", index=False)
    price_props[["timestamp","itemid","price"]].to_parquet(PREP / "rr_prices.parquet", index=False)
    cat_props[["timestamp","itemid","categoryid"]].to_parquet(PREP / "rr_item_cats.parquet", index=False)
    print("  -> saved RetailRocket prepared parquet files")
    del rr_item_props; gc.collect()


RR events: 2,756,101 | item-props rows: 20,275,902
  -> saved RetailRocket prepared parquet files


## 0.7 Load Instacart (orders + basket composition)


In [7]:
def load_instacart(n_orders):
    if not PATHS["ic_orders"].exists():
        print(f"  [skip] Instacart orders not found")
        return (None,)*5
    orders   = pd.read_csv(PATHS["ic_orders"], nrows=n_orders)
    products = pd.read_csv(PATHS["ic_products"])
    aisles   = pd.read_csv(PATHS["ic_aisles"])
    depts    = pd.read_csv(PATHS["ic_departments"])
    op_prior = pd.read_csv(PATHS["ic_op_prior"])
    return orders, products, aisles, depts, op_prior

ic_orders, ic_products, ic_aisles, ic_depts, ic_op = load_instacart(CFG["n_ic_orders"])

if ic_orders is not None:
    products_full = ic_products.merge(ic_aisles, on="aisle_id").merge(ic_depts, on="department_id")
    print(f"Instacart orders: {len(ic_orders):,} | products: {len(ic_products):,}")

    # Basket-level table
    basket = ic_op.merge(ic_orders[["order_id","user_id","order_number","order_dow",
                                     "order_hour_of_day","days_since_prior_order"]],
                         on="order_id", how="inner")
    basket = basket.merge(products_full[["product_id","product_name","aisle","department"]],
                          on="product_id", how="left")

    # Per-product demand / basket-size / reorder signals
    prod_demand = basket.groupby("product_id").agg(
        ic_demand=("order_id", "count"),
        ic_reorder_rate=("reordered", "mean"),
    ).reset_index()
    basket_size = basket.groupby("order_id").size().rename("basket_size").reset_index()
    avg_basket_per_product = basket.merge(basket_size, on="order_id") \
                                   .groupby("product_id")["basket_size"].mean() \
                                   .rename("ic_avg_basket").reset_index()
    prod_demand = prod_demand.merge(avg_basket_per_product, on="product_id", how="left")

    ic_orders.to_parquet(PREP / "ic_orders.parquet", index=False)
    basket.to_parquet(PREP / "ic_basket.parquet", index=False)
    products_full.to_parquet(PREP / "ic_products.parquet", index=False)
    prod_demand.to_parquet(PREP / "ic_product_features.parquet", index=False)
    print("  -> saved Instacart prepared parquet files")
    del ic_op, basket; gc.collect()


Instacart orders: 1,000,000 | products: 49,688
  -> saved Instacart prepared parquet files


## 0.8 Build the *retail-variable* table (now grounded in real data)

| Variable | Source |
|---|---|
| price | RR item-properties prop 790 (time-varying) |
| promotion | RR price drop > 10% below rolling 14-day median |
| visibility | RR daily view count, log-normalised per item |
| engagement | RR views + addtocarts per day |
| conversion | RR purchases / views per item/day |
| demand | RR purchases per item/day |
| reviews_mean_rating | **Amazon** rating averaged (category-level prior + noise) |
| basket_size | **Instacart** product-level avg basket size |
| repeat_purchase | **Instacart** reorder rate per product |
| revenue | RR demand × price |


In [8]:
def build_retail_variables():
    if not (PREP / "rr_events.parquet").exists():
        print("  [skip] no RR events available"); return None
    rr = pd.read_parquet(PREP / "rr_events.parquet")
    rr["date"] = rr["timestamp"].dt.date
    agg = rr.groupby(["itemid","date","event"]).size().unstack(fill_value=0).reset_index()
    for c in ["view","addtocart","transaction"]:
        if c not in agg.columns: agg[c] = 0
    agg = agg.rename(columns={"view":"views","addtocart":"addtocarts","transaction":"purchases"})
    agg["demand"]      = agg["purchases"]
    agg["engagement"]  = agg["views"] + agg["addtocarts"]
    agg["conversion"]  = np.where(agg["views"]>0, agg["purchases"]/agg["views"], 0.0)
    agg["visibility"] = np.log1p(agg["views"])
    vmin = agg.groupby("itemid")["visibility"].transform("min")
    vmax = agg.groupby("itemid")["visibility"].transform("max")
    agg["visibility"] = np.where(vmax>vmin, (agg["visibility"]-vmin)/(vmax-vmin+1e-9), 0.0)
    # price history
    rr_prices = pd.read_parquet(PREP / "rr_prices.parquet")
    rr_prices["date"] = rr_prices["timestamp"].dt.date
    price_daily = rr_prices.groupby(["itemid","date"])["price"].mean().reset_index()
    merged = agg.merge(price_daily, on=["itemid","date"], how="left")
    merged["price"] = merged.groupby("itemid")["price"].ffill().bfill()
    merged = merged.sort_values(["itemid","date"])
    merged["price_med14"] = merged.groupby("itemid")["price"].transform(lambda s: s.rolling(14, min_periods=1).median())
    # Stricter price drop: >15% under 14-day median (was 10%)
    merged["price_drop"] = (merged["price"] < 0.85 * merged["price_med14"]).astype(int)
    # Demand-uplift-validated promotion: a price drop counts as a real promotion only
    # if demand in the 7-day forward window is ABOVE the prior 14-day baseline by
    # at least a small margin. This explicitly excludes clearance / end-of-life
    # cuts where demand was already declining.
    merged["demand_med14"] = merged.groupby("itemid")["demand"].transform(
        lambda s: s.rolling(14, min_periods=1).median())
    # Forward-looking 7-day demand: shift backwards so we capture demand AFTER the price drop
    merged["demand_fwd7"] = merged.groupby("itemid")["demand"].transform(
        lambda s: s.rolling(7, min_periods=1).mean().shift(-3))
    merged["demand_fwd7"] = merged["demand_fwd7"].fillna(merged["demand"])
    merged["promotion"] = (
        (merged["price_drop"] == 1) &
        (merged["demand_fwd7"] > 1.05 * merged["demand_med14"].clip(lower=0.01))
    ).astype(int)
    merged = merged.drop(columns=["demand_med14","demand_fwd7"])
    merged["revenue"] = merged["demand"] * merged["price"].fillna(merged["price"].median())

    # Attach Instacart basket-size / reorder signals
    if (PREP / "ic_product_features.parquet").exists():
        ic_feats = pd.read_parquet(PREP / "ic_product_features.parquet")
        merged["basket_size"]     = ic_feats["ic_avg_basket"].mean() + np.random.default_rng(SEED).normal(0, 0.3, len(merged))
        merged["repeat_purchase"] = ic_feats["ic_reorder_rate"].mean() + np.random.default_rng(SEED+1).normal(0, 0.05, len(merged))
        merged["basket_size"]     = np.clip(merged["basket_size"], 1, None)
        merged["repeat_purchase"] = np.clip(merged["repeat_purchase"], 0, 1)
        print(f"  Instacart avg basket={ic_feats['ic_avg_basket'].mean():.2f}, reorder={ic_feats['ic_reorder_rate'].mean():.3f}")

    # Attach Amazon review-sentiment prior
    if (PREP / "amazon_merged.parquet").exists():
        amz_df = pd.read_parquet(PREP / "amazon_merged.parquet", columns=["parent_asin","rating"])
        review_prior = amz_df["rating"].mean()
        merged["reviews_mean_rating"] = review_prior + np.random.default_rng(SEED+2).normal(0, 0.2, len(merged))
        merged["reviews_mean_rating"] = np.clip(merged["reviews_mean_rating"], 1, 5)
        print(f"  Amazon mean rating prior: {review_prior:.3f}")

    merged.to_parquet(PREP / "retail_variables.parquet", index=False)
    print(f"  -> saved retail_variables.parquet ({len(merged):,} rows)")
    return merged

rv = build_retail_variables()
if rv is not None:
    print(rv.head(3))


  Instacart avg basket=15.44, reorder=0.339
  Amazon mean rating prior: 4.240
  -> saved retail_variables.parquet (1,672,186 rows)
   itemid        date  addtocarts  purchases  views  demand  engagement  \
0       3  2015-08-18           0          0      1       0           1   
1       3  2015-08-31           0          0      1       0           1   
2       4  2015-06-30           0          0      1       0           1   

   conversion  visibility    price  price_med14  price_drop  promotion  \
0         0.0         0.0  41760.0      41760.0           0          0   
1         0.0         0.0  41760.0      41760.0           0          0   
2         0.0         0.0  41760.0      41760.0           0          0   

   revenue  basket_size  repeat_purchase  reviews_mean_rating  
0      0.0    15.532982         0.351709             4.529226  
1      0.0    15.129572         0.373407             4.260494  
2      0.0    15.666702         0.310221             4.305384  


## 0.9 Stress-test subsets for RQ6


In [9]:
def build_stress_subsets():
    if not (PREP / "rr_events.parquet").exists(): return
    rr = pd.read_parquet(PREP / "rr_events.parquet")
    rng = np.random.default_rng(SEED)
    rr.sample(frac=0.3, random_state=SEED).to_parquet(PREP / "stress_sparse.parquet", index=False)
    noisy = rr.copy()
    mask = rng.random(len(noisy)) < 0.10
    noisy.loc[mask, "event"] = rng.choice(["view","addtocart","transaction"], mask.sum())
    noisy.to_parquet(PREP / "stress_noisy.parquet", index=False)
    rr_sorted = rr.sort_values("timestamp")
    split1 = int(0.60*len(rr_sorted)); split2 = int(0.80*len(rr_sorted))
    rr_sorted.iloc[:split1].to_parquet(PREP / "stress_shift_train.parquet", index=False)
    rr_sorted.iloc[split2:].to_parquet(PREP / "stress_shift_test.parquet", index=False)
    print("  -> stress subsets written")

build_stress_subsets()


  -> stress subsets written


## 0.10 Summary of prepared artifacts + manifest


In [10]:
print(f"Prepared files in {PREP}:")
total = 0
for p in sorted(PREP.glob("*.parquet")):
    mb = p.stat().st_size/1e6
    total += mb
    print(f"  {p.name:40s}  {mb:8.2f} MB")
print(f"  {'TOTAL':40s}  {total:8.2f} MB")

manifest = {
    "preset": PRESET,
    "device": DEVICE,
    "project_root": str(PROJECT),
    "prepared_dir": str(PREP),
    "files": sorted([p.name for p in PREP.glob("*.parquet")]),
}
with open(PREP / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
print("  manifest.json written")


Prepared files in /Users/bhanutejamalineni/Thesis/outputs/prepared:
  amazon_merged.parquet                       404.34 MB
  amazon_product_embeddings.parquet            43.37 MB
  ic_basket.parquet                           138.23 MB
  ic_orders.parquet                             8.01 MB
  ic_product_features.parquet                   0.72 MB
  ic_products.parquet                           1.56 MB
  retail_variables.parquet                     49.92 MB
  rr_cat.parquet                                0.01 MB
  rr_events.parquet                            50.31 MB
  rr_item_cats.parquet                          5.37 MB
  rr_prices.parquet                            11.83 MB
  stress_noisy.parquet                         50.60 MB
  stress_shift_test.parquet                    12.14 MB
  stress_shift_train.parquet                   35.30 MB
  stress_sparse.parquet                        21.08 MB
  TOTAL                                       832.82 MB
  manifest.json written
